# 🐍 Clase 9 · Construir Market y su API

> Construir el objeto que mantiene el índice temporal, reconstruye el OrderBook actual y delega cada orden al MatchingEngine.

**Hoy construyes:** Market: componer snapshots, OrderBook, MatchingEngine y tiempo.

⏱️ 🟢 LIVE ~20 min · 🔵 REQUIRED +20 min.

### Cómo funciona este cuaderno

Cada ejercicio declara una ruta pedagógica, decidida por contenido y no por posición: **🟢 LIVE** (núcleo presencial) · **🔵 REQUIRED** (consolidación autónoma requerida y evaluable) · **🟣 OPTIONAL** (profundización no obligatoria y no evaluable). Escribe tu respuesta, ejecuta la **✅ comprobación plegada** con `Shift+Enter` y usa la pista o solución solo cuando la necesites.

### B1 · Estado inicial

<sub>🟢 LIVE · núcleo presencial · ~4 min</sub>

Implementa `__init__`: conserva symbol/snapshots/depth, crea el engine y deja `_i=-1`, `book=None`.

<sub>practicas: __init__ y composición</sub>

In [ ]:
from exchange.matching import MatchingEngine
class StudentMarket:
    def __init__(self, symbol, snapshots, depth=10):
        pass
market = StudentMarket('BTC', [{'x':1}], 5)

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.__init__.__code__.co_consts != (None,), '⏸ implementa StudentMarket.__init__: su cuerpo sigue siendo pass'
assert market.symbol=='BTC' and market._snapshots==[{'x':1}] and market._depth==5
assert market._i==-1 and market.book is None
assert isinstance(market._engine,MatchingEngine)
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self,symbol,snapshots,depth=10):
        self.symbol=symbol; self._snapshots=snapshots; self._depth=depth
        self._engine=MatchingEngine(); self._i=-1; self.book=None
market=StudentMarket('BTC',[{'x':1}],5)
```

</details>

### B2 · Implementa step()

<sub>🟢 LIVE · núcleo presencial · ~6 min</sub>

Avanza un snapshot y reconstruye `book` con `OrderBook.from_snapshot`. Dos llamadas deben producir dos libros distintos.

<sub>practicas: cursor + factory de L7</sub>

In [ ]:
from exchange.book import OrderBook
from exchange.matching import MatchingEngine
rows=[{'bid_price_1':100,'bid_size_1':1,'ask_price_1':101,'ask_size_1':1}, {'bid_price_1':102,'bid_size_1':2,'ask_price_1':103,'ask_size_1':2}]
class StudentMarket:
    def __init__(self,symbol,snapshots,depth=10):
        self.symbol=symbol; self._snapshots=snapshots; self._depth=depth
        self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.step.__code__.co_consts != (None,), '⏸ implementa StudentMarket.step: su cuerpo sigue siendo pass'
m=StudentMarket('BTC',rows,1); b0=m.step(); b1=m.step()
assert b0.best_bid==100 and b1.best_bid==102
assert b0 is not b1 and m.book is b1 and m._i==1
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self,symbol,snapshots,depth=10):
        self.symbol=symbol; self._snapshots=snapshots; self._depth=depth
        self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth)
        return self.book
```

</details>

### B3 · Final de datos

<sub>🟢 LIVE · núcleo presencial · ~4 min</sub>

Completa `step()` para que, al agotarse los snapshots, devuelva `None` y también deje `book=None`. Las llamadas posteriores siguen siendo seguras.

<sub>practicas: estado terminal explícito</sub>

In [ ]:
from exchange.book import OrderBook
from exchange.matching import MatchingEngine
row={'bid_price_1':100,'bid_size_1':1,'ask_price_1':101,'ask_size_1':1}
class StudentMarket:
    def __init__(self):
        self.symbol='BTC'; self._snapshots=[row]; self._depth=1; self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.step.__code__.co_consts != (None,), '⏸ implementa StudentMarket.step: su cuerpo sigue siendo pass'
m=StudentMarket(); assert m.step() is not None
assert m.step() is None and m.book is None
assert m.step() is None and m.book is None
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self):
        self.symbol='BTC'; self._snapshots=[row]; self._depth=1; self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth)
        return self.book
```

</details>

### B4 · submit() sin book: fail fast

<sub>🟢 LIVE · núcleo presencial · ~3 min</sub>

Implementa el guard de `submit`: si no hay libro activo, lanza `RuntimeError` con un mensaje útil.

<sub>practicas: raise RuntimeError</sub>

In [ ]:
from exchange.matching import MatchingEngine
class StudentMarket:
    def __init__(self): self.book=None; self._engine=MatchingEngine()
    def submit(self,order):
        pass
market=StudentMarket()

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.submit.__code__.co_consts != (None,), '⏸ implementa StudentMarket.submit: su cuerpo sigue siendo pass'
try:
 market.submit(object())
 assert False,'debía lanzar RuntimeError'
except RuntimeError as e:
 assert 'step' in str(e).lower() or 'libro' in str(e).lower()
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self): self.book=None; self._engine=MatchingEngine()
    def submit(self,order):
        if self.book is None: raise RuntimeError('no hay libro activo: llama a step() primero')
        return self._engine.process(order,self.book,None)
market=StudentMarket()
```

</details>

### B5 · Delega al MatchingEngine

<sub>🟢 LIVE · núcleo presencial · ~3 min</sub>

Completa `submit`: debe llamar exactamente al engine contenido con `(order, book, timestamp)` y devolver su resultado.

<sub>practicas: composición, no duplicación</sub>

In [ ]:
class SpyEngine:
    def __init__(self): self.calls=[]
    def process(self,*args): self.calls.append(args); return ['fill']
class StudentMarket:
    def __init__(self):
        self.book=object(); self._engine=SpyEngine(); self._i=0; self._snapshots=[{'timestamp':123}]
    @property
    def timestamp(self): return self._snapshots[self._i]['timestamp']
    def submit(self,order):
        pass
market=StudentMarket(); order=object(); result=market.submit(order)

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.submit.__code__.co_consts != (None,), '⏸ implementa StudentMarket.submit: su cuerpo sigue siendo pass'
assert result==['fill']
assert market._engine.calls==[(order,market.book,123)]
print('ok — Market delega')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self):
        self.book=object(); self._engine=SpyEngine(); self._i=0; self._snapshots=[{'timestamp':123}]
    @property
    def timestamp(self): return self._snapshots[self._i]['timestamp']
    def submit(self,order):
        if self.book is None: raise RuntimeError('no hay libro activo: llama a step() primero')
        return self._engine.process(order,self.book,self.timestamp)
market=StudentMarket(); order=object(); result=market.submit(order)
```

</details>

### B6 · reset()

<sub>🔵 REQUIRED · consolidación requerida · ~6 min</sub>

Implementa `reset`. Después de avanzar y resetear, el primer `step()` debe reproducir exactamente el primer snapshot.

<sub>practicas: restaurar invariantes</sub>

In [ ]:
from exchange.book import OrderBook
from exchange.matching import MatchingEngine
rows=[{'bid_price_1':100,'bid_size_1':1,'ask_price_1':101,'ask_size_1':1},{'bid_price_1':102,'bid_size_1':1,'ask_price_1':103,'ask_size_1':1}]
class StudentMarket:
    def __init__(self): self.symbol='BTC'; self._snapshots=rows; self._depth=1; self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth); return self.book
    def reset(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert StudentMarket.reset.__code__.co_consts != (None,), '⏸ implementa StudentMarket.reset: su cuerpo sigue siendo pass'
m=StudentMarket(); first=m.step().best_bid; m.step(); m.reset()
assert m._i==-1 and m.book is None
assert m.step().best_bid==first==100
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class StudentMarket:
    def __init__(self): self.symbol='BTC'; self._snapshots=rows; self._depth=1; self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth); return self.book
    def reset(self):
        self._i=-1; self.book=None
```

</details>

### B7 · Ahora sí: loop completo

<sub>🔵 REQUIRED · consolidación requerida · ~6 min</sub>

Recorre `StudentMarket` hasta `None` y guarda los mids. El while aparece después de comprender el objeto que lo alimenta.

<sub>practicas: step hasta None</sub>

In [ ]:
from exchange.book import OrderBook
from exchange.matching import MatchingEngine
from exchange.market import Market
rows=Market.sample().snapshots[:12]
class StudentMarket:
    def __init__(self,symbol,rows,depth=3): self.symbol=symbol; self._snapshots=rows; self._depth=depth; self._engine=MatchingEngine(); self._i=-1; self.book=None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth); return self.book
market=StudentMarket('BTCUSDT',rows)
mids=[]

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert len(mids)==12 and all(isinstance(x,float) for x in mids)
assert market.book is None
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
market=StudentMarket('BTCUSDT',rows)
mids=[]
while True:
    book=market.step()
    if book is None: break
    mids.append(book.mid)
```

</details>

### B8 · Challenge de integración

<sub>🔵 REQUIRED · consolidación requerida · ~8 min</sub>

Integra las tres lessons: avanza, envía una MARKET, aplica fills al tracker, avanza otra vez y calcula equity. Guarda también `owners`, indicando qué objeto modifica cada acción.

<sub>practicas: step → submit → fills → tracker → equity</sub>

In [ ]:
from exchange.book import OrderBook
from exchange.matching import MatchingEngine
from exchange.market import Market
from exchange.orders import Order,OrderType
from exchange.portfolio import PositionTracker
rows=Market.sample().snapshots[:2]
class StudentMarket:
    def __init__(self): self.symbol='BTCUSDT'; self._snapshots=rows; self._depth=5; self._engine=MatchingEngine(); self._i=-1; self.book=None
    @property
    def timestamp(self): return int(self._snapshots[self._i].get('timestamp',self._i)) if 0<=self._i<len(self._snapshots) else None
    def step(self):
        self._i+=1
        if self._i>=len(self._snapshots): self.book=None; return None
        self.book=OrderBook.from_snapshot(self.symbol,self._snapshots[self._i],self._depth); return self.book
    def submit(self,o):
        if self.book is None: raise RuntimeError('llama a step primero')
        return self._engine.process(o,self.book,self.timestamp)
market=StudentMarket(); tracker=PositionTracker()
equity=None
owners=[]

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert equity is not None, '⏸ equity sigue en None: completa el ejercicio antes de validar'
assert isinstance(equity,float) and tracker.position>0
assert owners==['Market','MatchingEngine/OrderBook','PositionTracker','Market','PositionTracker']
print('ok  equity=%.2f'%equity)

<details>
<summary>💡 Ver solución</summary>

```python
market=StudentMarket(); tracker=PositionTracker(); owners=[]
book=market.step(); owners.append('Market')
fills=market.submit(Order('BTCUSDT','buy',.2,order_type=OrderType.MARKET)); owners.append('MatchingEngine/OrderBook')
for fill in fills: tracker.apply_fill(fill)
owners.append('PositionTracker')
book=market.step(); owners.append('Market')
equity=tracker.equity(book.mid); owners.append('PositionTracker')
```

</details>

## Cierre

Market añade tiempo y composición: step cambia el estado; submit delega la dinámica; reset vuelve al origen.

Cada ejercicio lleva una ruta explícita: LIVE, REQUIRED u OPTIONAL. Sigue REQUIRED para el itinerario autónomo y elige OPTIONAL solo si tienes margen.

**L10:** seguimos construyendo el sistema sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`run_day.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python run_day.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python run_day.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.